> **IMPORTS**

In [18]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import shutil

from Configuracion import CARGADOS, ANALITICOS_PARQUET, ANALITICOS_RESULTADOS, CARGA_GEO, RUTA_HADOOP, RECURSOS

RUTA_PARQUET_GUIA3 = ANALITICOS_PARQUET / "dataset_enriquecido_final.parquet"
RUTA_PARQUET_GEO= CARGA_GEO / "dataset_carga_geo.csv"

#Paquetes csv obtenidos publicamente en https://db-ip.com/ 
#IP geolocation API and database para cruzar datos relacionados

RUTA_CSV_CITY_LITE= RECURSOS /'dbip-city-lite-2026-10.csv.gz'
RUTA_CSV_ASN_LITE = RECURSOS /'dbip-asn-lite-2026-10.csv.gz'


>**SESION DE SPARK**

In [2]:
# --- Sesión de Spark ---
spark = (
    SparkSession.builder
        
    .appName("CybersecurityDataAnalytics")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "10")
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

print("Spark:", spark.version)
print("Paralelismo por defecto:", spark.sparkContext.defaultParallelism)


Spark: 3.5.3
Paralelismo por defecto: 12


# Datos unicos de tipo de IP

Se usan dos expresiones regulares (es_ipv4, no_publica)   para separar las IPs que se pueden geolocalizar de las que no.

**`es_ipv4`**: `^\d{1,3}(\.\d{1,3}){3}$`

Acepta solo texto con formato IPv4: cuatro grupos de 1 a 3 dígitos separados por puntos (por ejemplo, `69.28.157.213`). Descarta valores nulos, IPv6 y texto con otro formato.
Limitación: valida el formato, no que cada número esté entre 0 y 255.

**`no_publica`**: IPs que no existen en internet público y no se pueden geolocalizar.

esta acepta varios formatos que permiten identificar los tipos de conexion, Red privada (RFC 1918), Loopback, Autoasignada, Multicast , reservadas, broadcast y dentro del rango de direcciones especiales

Una IP es pública si cumple `es_ipv4` y no cumple `no_publica`.

>**CARGA INICIAL DE DATASET**

In [3]:
#Revisar la capacidad de relacion de los datos

print("\nContenido PARQUET_GUIA3: ")
df_Enriquecido = spark.read.parquet(str(RUTA_PARQUET_GUIA3))
df_Enriquecido.select("source_ip", "destination_ip").printSchema()
print("Filas:", df_Enriquecido.count())

#Datos unicos de tipo de IP

es_ipv4    = r"^\d{1,3}(\.\d{1,3}){3}$"
no_publica = r"^(10\.|192\.168\.|172\.(1[6-9]|2\d|3[01])\.|127\.|169\.254\.|0\.|2(2[4-9]|3\d|4\d|5[0-5])\.)"

ips = (df_Enriquecido.select(F.col("source_ip").alias("ip"))
         .union(df_Enriquecido.select(F.col("destination_ip").alias("ip")))
         .distinct())

ips_publicas_unicas = ips.filter(F.col("ip").rlike(es_ipv4) & ~F.col("ip").rlike(no_publica))

print("IPs distintas (total):", ips.count())
print("IPs públicas distintas:", ips_publicas_unicas.count())





Contenido PARQUET_GUIA3: 
root
 |-- source_ip: string (nullable = true)
 |-- destination_ip: string (nullable = true)

Filas: 2827562
IPs distintas (total): 19129
IPs públicas distintas: 19035


# Filtrado de las ip totales

`priv`: redes privadas (RFC 1918), usadas dentro de una organización, como las del laboratorio. Se repiten en muchas redes, por eso no se pueden ubicar geográficamente.

`espec`: direcciones de uso especial que no corresponden a un destino real en internet: loopback (la propia máquina), autoasignadas, multicast, reservadas y broadcast.

Con ambas, cada IP se clasifica como `privada`, `especial`, `publica` o `no_ipv4`. Solo las públicas se geolocalizan.

In [4]:
priv  = r"^(10\.|192\.168\.|172\.(1[6-9]|2\d|3[01])\.)"
espec = r"^(127\.|169\.254\.|0\.|2(2[4-9]|3\d|4\d|5[0-5])\.)"

def tipo_ip(c):
    col = F.col(c)
    return (F.when(col.isNull() | ~col.rlike(es_ipv4), "no_ipv4")
             .when(col.rlike(priv), "privada")
             .when(col.rlike(espec), "especial")
             .otherwise("publica"))

# 1) IPs distintas por tipo (source y destination juntas)
ips.withColumn("tipo", tipo_ip("ip")).groupBy("tipo").count().show()

# 2) Flujos según el tipo de cada extremo
df_Enriquecido_t = (df_Enriquecido
        .withColumn("Tipo_Emisor", tipo_ip("source_ip"))
        .withColumn("Tipo_Destinatario", tipo_ip("destination_ip"))
        .cache())
df_Enriquecido_t.groupBy("Tipo_Emisor", "Tipo_Destinatario").count().orderBy(F.desc("count")).show()


# 3) Por etiqueta: cuántos flujos tienen al menos una IP pública
(df_Enriquecido_t
    .withColumn("tiene_publica", (F.col("Tipo_Emisor") == "publica") | (F.col("Tipo_Destinatario") == "publica"))
    .groupBy("label", "tiene_publica").count()
    .orderBy("label", "tiene_publica").show(60, truncate=False))

+--------+-----+
|    tipo|count|
+--------+-----+
| publica|19035|
| privada|   18|
|especial|   76|
+--------+-----+

+-----------+-----------------+-------+
|Tipo_Emisor|Tipo_Destinatario|  count|
+-----------+-----------------+-------+
|    privada|          privada|1700262|
|    privada|          publica| 762775|
|    publica|          privada| 361639|
|    privada|         especial|   1633|
|    publica|          publica|   1184|
|    publica|         especial|     68|
|   especial|         especial|      1|
+-----------+-----------------+-------+

+--------------------------+-------------+-------+
|label                     |tiene_publica|count  |
+--------------------------+-------------+-------+
|BENIGN                    |false        |1147333|
|BENIGN                    |true         |1123674|
|Bot                       |true         |1956   |
|DDoS                      |false        |128025 |
|DoS GoldenEye             |false        |10293  |
|DoS Hulk                  |fal

# Documentacion de los descubrimientos

**Primer conteo:**

Conteo de tipos de ip basandose en las reglas de filtrado, que ayudaran a relacionarse al momento de cruzar los datos con las tablas de referencia geografica, 
ademas reduce la cantidad de ip que pueden ser tomadas como invalidas para el cruce.

**Segundo conteo:**

Confirmacion del flujo y las respuestas de receptor y el emisario con los diferentes tipos de ip antes segmentados, dando mas informacion para crear los filtros 
para reducir el dataset a solo ip con posible geocalizacion por medio de las listas geograficas.
    
**Tercer conteo:**

Segmenta la ip publicas por los tipos de flujos registrados, ayudando a identificar la relacion que existe entre ip publicas y los datos sinteticos usados para generar el dataset,
muestra que la mayoria de ip publicas forman parte de los registros BENIGN o de trafico normal, siendo solamente Infiltration y Bot que cuentan con ip publicas en sus registros.

# Demostraciones

-> Existen mayoritariamente ip publicas repartidas entre los regsitros, siendo la menor parte las privadas y especiales, sin correlacion inicial.

-> Los resultados de los flujos son: (Pulicos: 1,125,666 de 2,827,562 Flujos registrados) La IP pública aparece tanto en el origen como en el destino, por lo que hay que revisar las dos columnas para ubicar a la IP externa. Esto permite reducir el dataset a los flujos con posible geolocalización con filtros.

-> La gran mayoria de flujos son BENIGN, Solo Bot e Infiltration tienen flujos de ataque con IP pública; el resto de los ataques ocurre únicamente entre IPs privadas.

>**REDUCCION DE DATASET PARA ANALISIS**

In [5]:

df_carga_geo = (df_Enriquecido_t
    .filter((F.col("Tipo_Emisor") == "publica") | (F.col("Tipo_Destinatario") == "publica"))
    .select(
        F.col("source_ip").alias("ip_origen"),
        F.col("destination_ip").alias("ip_destino"),
        F.col("protocol").alias("protocolo"),
        F.col("timestamp").alias("fecha_hora"),
        F.col("label").alias("tipo_etiquetado"),
        F.col("flow_duration").alias("duracion_flujo"),
        F.col("total_fwd_packets").alias("paquetes_enviados"),
        F.col("total_backward_packets").alias("paquetes_recibidos"),
        F.col("Tipo_Emisor").alias("tipo_ip_origen"),
        F.col("Tipo_Destinatario").alias("tipo_ip_destino"),
    ))

df_carga_geo.show(3, truncate=False)

print("\nDescripcion de la tabla:")
print(f"\nFilas: {df_carga_geo.count()}")
print(f"\nColummas: {len(df_carga_geo.columns)}")

df_carga_geo.printSchema()

+-------------+-------------+---------+-------------------+---------------+--------------+-----------------+------------------+--------------+---------------+
|ip_origen    |ip_destino   |protocolo|fecha_hora         |tipo_etiquetado|duracion_flujo|paquetes_enviados|paquetes_recibidos|tipo_ip_origen|tipo_ip_destino|
+-------------+-------------+---------+-------------------+---------------+--------------+-----------------+------------------+--------------+---------------+
|69.28.157.213|192.168.10.14|6        |2017-07-05 01:55:00|BENIGN         |201.0         |3                |0                 |publica       |privada        |
|192.168.10.8 |104.31.91.87 |6        |2017-07-07 04:57:00|BENIGN         |209981.0      |15               |15                |privada       |publica        |
|192.168.10.15|34.196.21.126|6        |2017-07-04 03:46:00|BENIGN         |1.0380238E7   |9                |6                 |privada       |publica        |
+-------------+-------------+---------+-------

In [6]:
#Nulos

print("Nulos en campos criticos: ")
df_antes = df_carga_geo.count()

df_carga_geo = df_carga_geo.filter(
    F.col("fecha_hora").isNotNull() & F.col("tipo_etiquetado").isNotNull() & F.col("ip_origen").isNotNull() & F.col("ip_destino").isNotNull()
)
print(f"df_carga_geo: {df_antes:,} -> {df_carga_geo.count():,} filas ({df_antes - df_carga_geo.count():,} filas sin campos criticos eliminadas)")


Nulos en campos criticos: 
df_carga_geo: 1,125,666 -> 1,125,666 filas (0 filas sin campos criticos eliminadas)


> **LECTURA DE TABLA DE REFRENCIA GEOGRAFICA**

In [7]:
#Columnaas estipuldas en la pagina que provee los datos

cols_city = ["ip_inicio", "ip_fin", "continente", "pais", "estado", "ciudad", "latitud", "longitud"]
cols_asn  = ["ip_inicio", "ip_fin", "asn", "organizacion"]

#lectura de los archivos comprimidos
df_Listado_city = spark.read.csv(str(RUTA_CSV_CITY_LITE), header=False, inferSchema=False).toDF(*cols_city)
df_Listado_asn  = spark.read.csv(str(RUTA_CSV_ASN_LITE),  header=False, inferSchema=False).toDF(*cols_asn)

df_Listado_city.show(5, truncate=False)
df_Listado_asn.show(5, truncate=False)
print("\ndf_Listado_city: ")
print(df_Listado_city.columns)
print("\ndf_Listado_asn: ")
print(df_Listado_asn.columns)
print("")
df_Listado_city.printSchema()
print("")
df_Listado_asn.printSchema()

+---------+-------------+----------+----+----------+--------------+--------+--------+
|ip_inicio|ip_fin       |continente|pais|estado    |ciudad        |latitud |longitud|
+---------+-------------+----------+----+----------+--------------+--------+--------+
|0.0.0.0  |0.255.255.255|ZZ        |ZZ  |NULL      |NULL          |0       |0       |
|1.0.0.0  |1.0.0.255    |OC        |AU  |Queensland|South Brisbane|-27.4767|153.017 |
|1.0.1.0  |1.0.3.255    |AS        |CN  |Fujian    |Wenquan       |26.0998 |119.297 |
|1.0.4.0  |1.0.7.255    |OC        |AU  |Victoria  |Narre Warren  |-38.0268|145.301 |
|1.0.8.0  |1.0.15.255   |AS        |CN  |Guangdong |Guangzhou     |23.1317 |113.266 |
+---------+-------------+----------+----+----------+--------------+--------+--------+
only showing top 5 rows

+---------+-----------+-----+----------------------------+
|ip_inicio|ip_fin     |asn  |organizacion                |
+---------+-----------+-----+----------------------------+
|1.0.0.0  |1.0.0.255  |1

>**NORMALIZACION DE CAMPOS PARA JOIN**

In [8]:
#Convierte una IP escrita como texto en un número entero.
def ip_a_num(c):
    p = F.split(F.col(c), r"\.")
    return (p.getItem(0).cast("long") * 16777216 + p.getItem(1).cast("long") * 65536 +
            p.getItem(2).cast("long") * 256 + p.getItem(3).cast("long"))


city = (df_Listado_city.filter(F.col("ip_inicio").rlike(es_ipv4)) #aplicar regla crreada anteriormente para filtrar solo ip IPv4
        .withColumn("ip_ini_num", ip_a_num("ip_inicio")) #aplicar funcion 
        .withColumn("ip_fin_num", ip_a_num("ip_fin")) #aplicar funcion
        .withColumn("latitud", F.col("latitud").cast("double")) #cambio de formato string a double
        .withColumn("longitud", F.col("longitud").cast("double"))) #cambio de formato string a double

asn = (df_Listado_asn.filter(F.col("ip_inicio").rlike(es_ipv4))
        .withColumn("ip_ini_num", ip_a_num("ip_inicio")) #aplicar funcion
        .withColumn("ip_fin_num", ip_a_num("ip_fin"))) #aplicar funcion

print("City IPv4:", city.count(), "| ASN IPv4:", asn.count())
city.show(5, truncate=False)
asn.show(5, truncate=False)

City IPv4: 3620434 | ASN IPv4: 402389
+---------+-------------+----------+----+----------+--------------+--------+--------+----------+----------+
|ip_inicio|ip_fin       |continente|pais|estado    |ciudad        |latitud |longitud|ip_ini_num|ip_fin_num|
+---------+-------------+----------+----+----------+--------------+--------+--------+----------+----------+
|0.0.0.0  |0.255.255.255|ZZ        |ZZ  |NULL      |NULL          |0.0     |0.0     |0         |16777215  |
|1.0.0.0  |1.0.0.255    |OC        |AU  |Queensland|South Brisbane|-27.4767|153.017 |16777216  |16777471  |
|1.0.1.0  |1.0.3.255    |AS        |CN  |Fujian    |Wenquan       |26.0998 |119.297 |16777472  |16778239  |
|1.0.4.0  |1.0.7.255    |OC        |AU  |Victoria  |Narre Warren  |-38.0268|145.301 |16778240  |16779263  |
|1.0.8.0  |1.0.15.255   |AS        |CN  |Guangdong |Guangzhou     |23.1317 |113.266 |16779264  |16781311  |
+---------+-------------+----------+----+----------+--------------+--------+--------+----------+--

>**JOIN CON LAS IPS UNICAS OBTENIDAS**

In [9]:
#ips unicas registradas en ejecucion
ips_num = ips_publicas_unicas.withColumn("ip_num", ip_a_num("ip"))
ips_b = F.broadcast(ips_num.withColumn("bloque", F.floor(F.col("ip_num") / 65536).cast("long")))


# 2) Cada rango se reparte en los bloques que cubre, y se cruza solo con las IPs de ese bloque
def cruzar(tabla, cols):
    t = tabla.withColumn("bloque", F.explode(F.sequence(
            F.floor(F.col("ip_ini_num") / 65536).cast("long"),
            F.floor(F.col("ip_fin_num") / 65536).cast("long"))))
    return (t.join(ips_b, "bloque")
             .filter((F.col("ip_num") >= F.col("ip_ini_num")) & (F.col("ip_num") <= F.col("ip_fin_num")))
             .select("ip", *cols))

ip_city = cruzar(city, ["continente", "pais", "estado", "ciudad", "latitud", "longitud"])
ip_asn  = cruzar(asn,  ["asn", "organizacion"])

# 3) Una fila por IP, con los campos geográficos
ip_geo = ips_num.select("ip").join(ip_city, "ip", "left").join(ip_asn, "ip", "left").cache()


print("Filas:", ip_geo.count(), "| IPs distintas:", ip_geo.select("ip").distinct().count())
print("Sin país:", ip_geo.filter(F.col("pais").isNull()).count())
ip_geo.show(5, truncate=False)


Filas: 19035 | IPs distintas: 19035
Sin país: 0
+--------------+----------+----+----------+-----------------+-------+--------+-----+------------------+
|ip            |continente|pais|estado    |ciudad           |latitud|longitud|asn  |organizacion      |
+--------------+----------+----+----------+-----------------+-------+--------+-----+------------------+
|52.84.145.167 |NA        |US  |Minnesota |Minneapolis      |44.9778|-93.265 |16509|Amazon.com, Inc.  |
|142.0.160.53  |NA        |US  |Virginia  |Ashburn          |39.0418|-77.4744|31898|Oracle Corporation|
|35.157.81.84  |EU        |DE  |Hesse     |Frankfurt am Main|50.1109|8.68213 |16509|Amazon.com, Inc.  |
|54.192.37.254 |NA        |US  |California|San Jose         |37.3387|-121.885|16509|Amazon.com, Inc.  |
|151.101.20.166|NA        |US  |Washington|Seattle          |47.6061|-122.333|54113|Fastly, Inc.      |
+--------------+----------+----+----------+-----------------+-------+--------+-----+------------------+
only showing top

>**REVISION DE CALIDAD**

In [10]:
print("\nCampos criticos:\n")
total_ips = ip_geo.count()
for c in ["estado", "ciudad", "organizacion"]:
    n = ip_geo.filter(F.col(c).isNull() | (F.trim(F.col(c)) == "")).count()
    print(f"{c}: {n:,} vacíos ({n/total_ips*100:.2f}%)")
for c in ["latitud", "longitud", "asn"]:
    n = ip_geo.filter(F.col(c).isNull()).count()
    print(f"{c}: {n:,} nulos ({n/total_ips*100:.2f}%)")

ag = ip_geo.filter(F.col("asn").isNull() & F.col("organizacion").isNotNull()).count()
print("IPs con organización pero sin ASN (debe haber coincidencia seria 0 ):", ag)

print("\nIPs por país:")
ip_geo.groupBy("pais").count().orderBy(F.desc("count")).show(10)

print("IPs por organización:")
ip_geo.groupBy("organizacion").count().orderBy(F.desc("count")).show(10, truncate=False)


Campos criticos:

estado: 211 vacíos (1.11%)
ciudad: 0 vacíos (0.00%)
organizacion: 958 vacíos (5.03%)
latitud: 0 nulos (0.00%)
longitud: 0 nulos (0.00%)
asn: 958 nulos (5.03%)
IPs con organización pero sin ASN (debe haber coincidencia seria 0 ): 0

IPs por país:
+----+-----+
|pais|count|
+----+-----+
|  US|11021|
|  CA| 1658|
|  JP| 1210|
|  DE|  629|
|  IE|  559|
|  CN|  458|
|  RU|  399|
|  GB|  317|
|  IT|  309|
|  FR|  299|
+----+-----+
only showing top 10 rows

IPs por organización:
+-------------------------+-----+
|organizacion             |count|
+-------------------------+-----+
|Amazon.com, Inc.         |7747 |
|Akamai International B.V.|1670 |
|NULL                     |958  |
|Akamai Technologies, Inc.|926  |
|Google LLC               |836  |
|Cloudflare, Inc.         |538  |
|Fastly, Inc.             |397  |
|Rackspace Hosting        |305  |
|OVH SAS                  |285  |
|Microsoft Corporation    |264  |
+-------------------------+-----+
only showing top 10 rows



In [11]:
#normalizacion de nulos
ip_geo = (ip_geo
    .withColumn("estado", F.when(F.col("estado").isNull() | (F.trim(F.col("estado")) == ""), "Sin dato")
                           .otherwise(F.col("estado")))
    .withColumn("organizacion", F.coalesce(F.col("organizacion"), F.lit("Sin dato")))
    .withColumn("asn", F.coalesce(F.col("asn"), F.lit("Sin dato")))
    .cache())

print("Sin dato estipulado en organizacion:", ip_geo.filter(F.col("organizacion") == "Sin dato").count())
print("Sin dato estipulado en estado:", ip_geo.filter(F.col("estado") == "Sin dato").count())
print("Sin dato estipulado en asn:", ip_geo.filter(F.col("asn") == "Sin dato").count())

Sin dato estipulado en organizacion: 958
Sin dato estipulado en estado: 211
Sin dato estipulado en asn: 958


>**JOIN (IPS CON LISTADO DE DIRECCIONES) + (FLUJOS DE DATAFRAME PROCESADO)**

In [20]:
def con_geo(df, col_ip, prefijo):
    g = ip_geo.select(
        F.col("ip").alias(col_ip),
        F.col("continente").alias(f"{prefijo}_continente"),
        F.col("pais").alias(f"{prefijo}_pais"),
        F.col("estado").alias(f"{prefijo}_estado"),
        F.col("ciudad").alias(f"{prefijo}_ciudad"),
        F.col("latitud").alias(f"{prefijo}_latitud"),
        F.col("longitud").alias(f"{prefijo}_longitud"),
        F.col("organizacion").alias(f"{prefijo}_organizacion"))
    return df.join(F.broadcast(g), col_ip, "left")

df_flujos_geo = con_geo(con_geo(df_carga_geo, "ip_origen", "origen"), "ip_destino", "destino")

>**COMPROBACIONES**

In [21]:
# Las filas deben ser las mismas antes y después reflenado si fueron duplicadas en el left
print("Filas antes:", df_carga_geo.count(), "| después:", df_flujos_geo.count())
print("Columnas:", len(df_flujos_geo.columns))

# Muestra de flujos completos con su ubicacion relacionada
(df_flujos_geo.filter(F.col("destino_pais").isNotNull())
    .select("ip_origen", "ip_destino", "destino_pais", "destino_ciudad", "destino_organizacion")
    .show(3, truncate=False))

# Flujos por país, en origen y en destino
df_flujos_geo.groupBy("destino_pais").count().orderBy(F.desc("count")).show(10)
df_flujos_geo.groupBy("origen_pais").count().orderBy(F.desc("count")).show(10)

print("Distribucion de ips no rastreables (nulas)")
print("Ambos nulos:", df_flujos_geo.filter(F.col("origen_pais").isNull() & F.col("destino_pais").isNull()).count())
print("Ninguno nulo:", df_flujos_geo.filter(F.col("origen_pais").isNotNull() & F.col("destino_pais").isNotNull()).count())

Filas antes: 1125666 | después: 1125666
Columnas: 24
+-------------+---------------+------------+--------------+--------------------+
|ip_origen    |ip_destino     |destino_pais|destino_ciudad|destino_organizacion|
+-------------+---------------+------------+--------------+--------------------+
|192.168.10.8 |104.31.91.87   |CA          |Montreal      |Sin dato            |
|192.168.10.15|34.196.21.126  |US          |Ashburn       |Amazon.com, Inc.    |
|192.168.10.8 |138.201.138.163|DE          |Falkenstein   |Hetzner Online GmbH |
+-------------+---------------+------------+--------------+--------------------+
only showing top 3 rows

+------------+------+
|destino_pais| count|
+------------+------+
|          US|494236|
|        NULL|361707|
|          CA| 76695|
|          GB| 30598|
|          JP| 26000|
|          DE| 16465|
|          RU| 10876|
|          FR| 10855|
|          KR|  9458|
|          CN|  9004|
+------------+------+
only showing top 10 rows

+-----------+------+


In [22]:
#eliminacion de regsitros con ambos lados rastreables para evitar solapamiento de los datos

#Fuera los flujos con ambos extremos públicos
df_flujos_geo = df_flujos_geo.filter(
    ~((F.col("tipo_ip_origen") == "publica") & (F.col("tipo_ip_destino") == "publica"))
)
print("Filas finales:", df_flujos_geo.count())


Filas finales: 1124482


>**CREACION DE VISTAS (CIUDADES, DIAS, DIRECCION, ATAQUES)**

In [23]:

df_flujos_geo = df_flujos_geo.filter(
    ~((F.col("tipo_ip_origen") == "publica") & (F.col("tipo_ip_destino") == "publica")))

df_flujos_geo = df_flujos_geo.filter(
    ~((F.col("tipo_ip_origen") == "publica") & (F.col("tipo_ip_destino") == "publica")))

def lado_externo(p):
    return df_flujos_geo.filter(F.col(f"tipo_ip_{p}") == "publica").select(
        F.col(f"{p}_continente").alias("continente"),
        F.col(f"{p}_pais").alias("pais"),
        F.col(f"{p}_ciudad").alias("ciudad"),
        F.col(f"{p}_latitud").alias("latitud"),
        F.col(f"{p}_longitud").alias("longitud"),
        F.col(f"{p}_organizacion").alias("organizacion"),
        (F.col("paquetes_enviados") + F.col("paquetes_recibidos")).alias("paquetes"),
        F.col("duracion_flujo"),
        F.col("tipo_etiquetado").alias("etiqueta"),
        (F.col("tipo_etiquetado") != "BENIGN").cast("int").alias("es_ataque"))

ext = lado_externo("destino").unionByName(lado_externo("origen")).cache()

geo = ["continente", "pais", "ciudad", "latitud", "longitud"]
medidas = [F.count("*").alias("flujos"), F.sum("paquetes").alias("paquetes")]

capas = {
    "capa_ciudades": ext.groupBy(*geo).agg(
        *medidas,
        F.sum("es_ataque").alias("flujos_ataque"),
        F.round(F.avg("duracion_flujo") / 1e6, 2).alias("duracion_prom_seg")),
    "capa_ataques": ext.filter(F.col("es_ataque") == 1).groupBy(*geo, "etiqueta").agg(*medidas),
    "capa_organizaciones": ext.groupBy(*geo, "organizacion").agg(*medidas),
}


>**VALIDACION DE COORDENADAS**

In [24]:
# --- Validación de coordenadas y de gobernanza ---
total_esperado = ext.count()
print("Flujos con extremo externo:", f"{ext.count():,}")
for nombre, df_c in capas.items():
    n = df_c.count()
    validas = df_c.filter(
        F.col("latitud").between(-90, 90) & F.col("longitud").between(-180, 180) &
        ~((F.col("latitud") == 0) & (F.col("longitud") == 0))).count()
    suma = df_c.agg(F.sum("flujos")).first()[0]
    assert not {"ip_origen", "ip_destino", "ip"} & set(df_c.columns), f"{nombre} trae IPs"
    print(f"{nombre}: {n:,} filas | coordenadas válidas: {validas:,} | a revisar: {n - validas:,} | suma flujos: {suma:,}")


Flujos con extremo externo: 1,124,482
capa_ciudades: 1,085 filas | coordenadas válidas: 1,085 | a revisar: 0 | suma flujos: 1,124,482
capa_ataques: 3 filas | coordenadas válidas: 3 | a revisar: 0 | suma flujos: 1,992
capa_organizaciones: 1,687 filas | coordenadas válidas: 1,687 | a revisar: 0 | suma flujos: 1,124,482


>**EXPORTACION DE DATASET CON CAPAS**

In [25]:
# --- Exportar ---
def exportar_csv(df, nombre):
    tmp = CARGA_GEO / f"_tmp_{nombre}"
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(tmp))
    shutil.copy(next(tmp.glob("part-*.csv")), CARGA_GEO / f"{nombre}.csv")
    shutil.rmtree(tmp)

for nombre, df_c in capas.items():
    exportar_csv(df_c.orderBy(F.desc("flujos")), nombre)

## Documentación de las capas

**Flujos con extremo externo**
Tras excluir los 1,184 flujos con ambas IPs públicas, quedan 1,124,482 flujos con un único extremo externo. A cada uno se le toma la ubicación del lado público, el destino si el tráfico sale del laboratorio y el origen si llega.

**Validación de las capas**

Se generaron tres capas a partir de `df_flujos_geo`, todas agregadas y sin columnas de IP como parte de la implementacion de reglas de gobernanza

`capa_ciudades`  Flujos, paquetes, flujos de ataque y duración promedio por ciudad

`capa_ataques`  Flujos y paquetes por ciudad y tipo de ataque 

`capa_organizaciones`: flujos y paquetes por ciudad y organización.


Las coordenadas se validaron con un rango global (latitud entre -90 y 90, longitud entre -180 y 180) y descartando el punto (0, 0). Las tres capas tienen 0 coordenadas a revisar.

**Totales de registros repartidos entre las capas:**
La suma de flujos de es 1,124,482 en las dos principales, igual al total con extremo externo. La suma de `capa_ataques` es 1,992, que coincide con cantidades de los flujos de Bot  e Infiltration identificados antes como unicos en ser registrados con ip publicas.


## Demostraciones

-> Los 1,124,482 flujos se agrupan en 1,085 ciudades, por lo que el mapa pasa de más de un millón de registros a un número de puntos manejable.

-> Ninguna de las tres capas tiene coordenadas fuera de rango, y ninguna incluye IPs que exponga datos sensibles, por lo que se pueden publicar.

-> Las tres capas completas suman exactamente el total de flujos, lo que indica que no se perdieron ni se duplicaron registros al agrupar.

-> Los flujos de ataque con IP pública se limitan a Bot e Infiltration con 1,992 flujos, repartidos en 3 combinaciones de ciudad y tipo de ataque. Es la capa mas reducida

-> La capa de organizaciones tiene n filas y muestra con qué organizaciones se comunicó la red, mostrando las principlas que concentran más flujos.
